# Serialization Options
Xopt objects serialize to YAML/JSON for checkpointing and restarts. What goes
into a dump is controlled by a few writer options, passed to `yaml()`, `json()`
or `dump()`:

- `module_mode`: `"drop"` (default), `"file"` (write torch modules as `.pt`
  sidecar files next to the dump file) or `"inline"` (embed as base64 strings)
- `array_mode`: `"list"` (default) or `"b64"` (binary numpy/torch payloads)
- `df_mode`: `"dict"` (default) or `"b64"` for dataframes
- `compress`: `None` (default), `"gzip"` or `"zstd"`, with an optional `level`

This example compares dump sizes for a sizeable Xopt instance.

In [ ]:
import math

from xopt import Xopt, Evaluator, VOCS
from xopt.generators.bayesian import UpperConfidenceBoundGenerator
from xopt.resources.test_functions.sinusoid_1d import evaluate_sinusoid

vocs = VOCS(variables={"x1": [0, 1.75 * math.pi]}, objectives={"y1": "MINIMIZE"})
X = Xopt(
    generator=UpperConfidenceBoundGenerator(vocs=vocs),
    evaluator=Evaluator(function=evaluate_sinusoid),
)

# evaluate 500 random points and train the GP model
X.random_evaluate(500)
X.generator.train_model()
X.generator.model

## Default dump
By default torch modules (like the trained GP model above) are dropped and
arrays/dataframes are written as plain lists and dicts.

In [ ]:
print(f"default: {len(X.yaml()) / 1024:.0f} KiB")

## Inline torch modules
With `module_mode="inline"` the trained model is embedded in the dump as a
base64 payload, so the file is fully self-contained. Compression shrinks the
binary payloads considerably; combining it with `b64` array and dataframe
modes compresses the evaluation data as well.

In [ ]:
import time

b64 = dict(module_mode="inline", array_mode="b64", df_mode="b64")
variants = {
    "inline raw (lists)": dict(module_mode="inline"),
    "inline b64": b64,
    "inline b64 + gzip 9": dict(**b64, compress="gzip", level=9),
    "inline b64 + zstd 3": dict(**b64, compress="zstd", level=3),
    "inline b64 + zstd 22 (max)": dict(**b64, compress="zstd", level=22),
}
for name, kwargs in variants.items():
    start = time.perf_counter()
    size = len(X.yaml(**kwargs))
    elapsed = time.perf_counter() - start
    print(f"{name:28s} {size / 1024:4.0f} KiB  {elapsed * 1e3:6.1f} ms")

When `level` is not given, gzip defaults to level 9 and zstd to the
zstandard default (level 3). Higher zstd levels trade dump time for
size.

## Reloading
Every variant reloads with `from_yaml`/`from_file`, including the trained
model.

In [ ]:
X2 = Xopt.from_yaml(X.yaml(module_mode="inline", compress="zstd"))
X2.generator.model

For long optimization runs prefer `module_mode="file"` (the default used by
`X.dump()` when `serialize_torch=True`): modules are written as `.pt` files
next to the dump file, and `Xopt.from_file` finds them from any working
directory.